# Week 2 Monday — RAG, planning, and a light multi-agent handoff

**DAT 5668 · Module 2, Session 1 · Fundamentals (about 80 minutes)**

**Family this session: LLM-based agent.** Retrieval, a plan, and a handoff across researcher, analyst, and reviewer. Wednesday switches to an RL-based agent.

![Researcher, analyst, and reviewer](../../assets/rag-handoff.png)

## Run of show
- 10 min — Recap. Cold call: what did the Week 1 agent observe that the chatbot skipped?
- 20 min — Chunks, retrieval, citations, and why a wrong chunk is a management problem.
- 25 min — Live demo of retrieval over a short policy set, then a three-role handoff.
- 15 min — Planning for a multi-step task. Preview Wednesday's bandit lab: a different family.

## Learning goals
- Retrieve from enterprise text with a citation, and spot a bad retrieval.
- Break a task into steps the agent can check.
- Hand work across researcher, analyst, and reviewer without losing accountability.


## 1. Why retrieval exists

A model that answers from memory will invent a refund window. Retrieval-augmented generation (RAG) forces the answer to come from **your** documents.

Instructional policy set (not a real firm):


In [ ]:
# Each string is one chunk. In a real stack you would split long files on purpose
# (headings, or a fixed token length) and keep the source name for the citation.
CHUNKS = [
    {"id": "P1", "source": "Refund policy", "text": "A refund for damaged items is allowed within 30 days of delivery. The amount is the order total."},
    {"id": "P2", "source": "Refund policy", "text": "A refund for a change of mind is not allowed. Offer store credit within 14 days instead."},
    {"id": "P3", "source": "Privacy", "text": "Do not paste customer addresses or payment numbers into a public chatbot."},
    {"id": "P4", "source": "Shipping", "text": "Lost shipments may be replaced after 7 business days. Refunds for lost shipments require a carrier trace."},
]


def tokens(text):
    return [word.strip(".,").lower() for word in text.split() if len(word) > 2]


def retrieve(query, k=2):
    """Score chunks by shared tokens. Simple on purpose so the failure mode is visible."""
    query_terms = set(tokens(query))
    scored = []
    for chunk in CHUNKS:
        overlap = query_terms & set(tokens(chunk["text"]))
        scored.append((len(overlap), chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


def answer(query):
    hits = retrieve(query, k=1)
    score, chunk = hits[0]
    if score == 0:
        return {"answer": "I cannot ground this. Escalate.", "citation": None, "score": 0}
    return {
        "answer": chunk["text"],
        "citation": f"{chunk['source']} ({chunk['id']})",
        "score": score,
    }


for query in [
    "Can we cash-refund a damaged mug after 10 days?",
    "Customer wants cash back because they changed their mind.",
    "What is our vendor's SOC 2 status?",
]:
    result = answer(query)
    print(f"Q: {query}")
    print(f"   score={result['score']} citation={result['citation']}")
    print(f"   {result['answer']}\n")


## 2. Retrieval errors are a management problem

The third query shares no policy language, so the agent refuses. That is the correct failure.

Now force a **bad chunk**. Overlap search will still fire on "refund" for a change-of-mind question and may surface the damaged-item rule first. If you show that sentence to a customer, you have an unapproved policy.

What managers should ask:

- Did the chunk come from the system of record, or from an old PDF?
- Does the reply cite the chunk id?
- What is the cost of the wrong chunk (cash out the door, privacy, a bad promise)?


In [ ]:
# Show the top two chunks so the class can see a near miss.
print("Near miss for a change-of-mind question:")
for score, chunk in retrieve("Is a refund allowed?", k=2):
    print(f"  score={score} {chunk['id']}: {chunk['text']}")


## 3. Plan, then hand off

Multi-step task: "Should we refund order 1003, and who signs?"

Roles, each accountable for one artifact:

| Role | Owns | Must pass on |
| --- | --- | --- |
| Researcher | The cited chunk | citation id or an explicit miss |
| Analyst | The recommendation | action plus the chunk it used |
| Reviewer | The release decision | approve only if the citation is real |


In [ ]:
def researcher(query):
    result = answer(query)
    return {"role": "researcher", "citation": result["citation"], "evidence": result["answer"]}


def analyst(research):
    if research["citation"] is None:
        action = "escalate"
    elif "30 days" in research["evidence"]:
        action = "apply the 30-day damage rule; do not invent a new window"
    else:
        action = "follow the cited sentence only"
    return {"role": "analyst", "action": action, "used": research["citation"]}


def reviewer(research, analysis):
    known_ids = {chunk["id"] for chunk in CHUNKS}
    cited = research["citation"] or ""
    has_real_source = any(chunk_id in cited for chunk_id in known_ids)
    approved = has_real_source and analysis["used"] == research["citation"]
    return {"role": "reviewer", "approved": approved, "reason": "citation matches the corpus" if approved else "missing citation"}


query = "Damaged item refund window"
research = researcher(query)
analysis = analyst(research)
review = reviewer(research, analysis)
print(research)
print(analysis)
print(review)

# A broken handoff: the analyst cites nothing. The reviewer must stop the release.
bad_analysis = {"role": "analyst", "action": "refund everything", "used": None}
print("Broken handoff:", reviewer(research, bad_analysis))


## 4. Bridge to Wednesday

Wednesday leaves documents and practices the other family: **repeated decisions under uncertainty**. A pricing test does not need a paragraph. It needs a choice among offers, a numeric reward, and a rule for how much to explore.

Students should leave Monday able to say: RAG is for knowledge work with a corpus; a bandit is for a repeated test with a payoff.


## Fill this with me

Write a customer question in your own words. Run it. If the citation is wrong or missing, say so. That is the management problem.


In [ ]:
# FILL WITH THE CLASS.
my_question = ""
if not my_question.strip():
    print("Type a question, then run this cell again.")
else:
    found = answer(my_question)
    print("Citation:", found["citation"])
    print(found["answer"])
    print("Did the sentence actually answer your question? Say yes or no out loud.")
